# Salary regression (ANN)

Predicts **EstimatedSalary** from other bank features. This notebook saves **only** `onehot_encoder_geo.pkl` (Geography `OneHotEncoder`). Gender is label-encoded in memory; the print shows class order for your app.

In [ ]:
import datetime
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

%matplotlib inline


In [ ]:
df = pd.read_csv('Churn_Modelling.csv')
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
df.head()


In [ ]:
label_encoder_gender = LabelEncoder()
df['Gender'] = label_encoder_gender.fit_transform(df['Gender'])
print('Gender label order:', list(label_encoder_gender.classes_))


In [ ]:
onehot_encoder_geo = OneHotEncoder()
geo_encoded = onehot_encoder_geo.fit_transform(df[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography']),
    index=df.index,
)
df = pd.concat([df.drop(columns=['Geography']), geo_encoded_df], axis=1)
df.head()


In [ ]:
X = df.drop(columns=['EstimatedSalary'])
y = df['EstimatedSalary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
with open('onehot_encoder_geo.pkl', 'wb') as f:
    pickle.dump(onehot_encoder_geo, f)
print('Saved: onehot_encoder_geo.pkl')


In [ ]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1),
])
model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae'])
model.summary()


In [ ]:
log_dir = 'regressionlogs/fit/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
tb_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping = EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)


In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping, tb_callback],
)


In [ ]:
%load_ext tensorboard
%tensorboard --logdir regressionlogs/fit


In [ ]:
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
print(f'Test MAE (loss): {test_loss:.2f}')


In [ ]:
model.save('regression_model.h5')
